In [ ]:
# Fill remaining NDVI missing values by seasonal median (same week in other years)
for city in ['sj', 'iq']:
    for ndvi_dir in ['ndvi_ne', 'ndvi_nw', 'ndvi_se', 'ndvi_sw']:
        df.loc[df['city'] == city, ndvi_dir] = df.loc[df['city'] == city].groupby('weekofyear')[ndvi_dir].transform(
            lambda x: x.fillna(x.median()))
        
        # If still missing values, use city median
        if df.loc[df['city'] == city, ndvi_dir].isna().sum() > 0:
            median_val = df.loc[df['city'] == city, ndvi_dir].median()
            df.loc[(df['city'] == city) & (df[ndvi_dir].isna()), ndvi_dir] = median_val

In [ ]:
# For station temperature variables
station_vars = ['station_diur_temp_rng_c', 'station_avg_temp_c', 'station_max_temp_c', 'station_min_temp_c']

for city in ['sj', 'iq']:
    city_mask = df['city'] == city
    
    for var in station_vars:
        # First try seasonal imputation (same week in other years)
        df.loc[city_mask, var] = df.loc[city_mask].groupby('weekofyear')[var].transform(
            lambda x: x.fillna(x.median()))
        
        # For any remaining missing values, use interpolation
        df.loc[city_mask, var] = df.loc[city_mask, var].interpolate(method='linear')
        
        # If still have missing values at the edges, use forward/backward fill
        df.loc[city_mask, var] = df.loc[city_mask, var].fillna(method='ffill').fillna(method='bfill')

In [ ]:
# For precipitation
for city in ['sj', 'iq']:
    city_mask = df['city'] == city
    # Use seasonal median first
    df.loc[city_mask, 'station_precip_mm'] = df.loc[city_mask].groupby('weekofyear')['station_precip_mm'].transform(
        lambda x: x.fillna(x.median()))

    # Fill any remaining with city median
    if df.loc[city_mask, 'station_precip_mm'].isna().sum() > 0:
        median_val = df.loc[city_mask, 'station_precip_mm'].median()
        df.loc[(city_mask) & (df['station_precip_mm'].isna()), 'station_precip_mm'] = median_val

In [ ]:
# For reanalysis variables with very low missing rates
reanalysis_vars = [col for col in df.columns if col.startswith('reanalysis_')]

for city in ['sj', 'iq']:
    city_mask = df['city'] == city
    
    for var in reanalysis_vars:
        # First try interpolation which works well for time series
        df.loc[city_mask, var] = df.loc[city_mask, var].interpolate(method='linear')
        
        # If still have missing values at the edges, use seasonal median
        if df.loc[city_mask, var].isna().sum() > 0:
            df.loc[city_mask, var] = df.loc[city_mask].groupby('weekofyear')[var].transform(
                lambda x: x.fillna(x.median()))
        
        # Finally, if any remain, use forward/backward fill
        df.loc[city_mask, var] = df.loc[city_mask, var].fillna(method='ffill').fillna(method='bfill')

In [ ]:
# For remaining precipitation measures
precip_vars = ['precipitation_amt_mm', 'reanalysis_sat_precip_amt_mm']

for city in ['sj', 'iq']:
    city_mask = df['city'] == city
    
    for var in precip_vars:
        # Use seasonal median
        df.loc[city_mask, var] = df.loc[city_mask].groupby('weekofyear')[var].transform(
            lambda x: x.fillna(x.median()))
        
        # Fill any remaining with city median
        if df.loc[city_mask, var].isna().sum() > 0:
            median_val = df.loc[city_mask, var].median()
            df.loc[(city_mask) & (df[var].isna()), var] = median_val

In [ ]:
# Verify temperature consistency
temp_check = df[(df['station_min_temp_c'] > df['station_avg_temp_c']) | 
                (df['station_avg_temp_c'] > df['station_max_temp_c'])]

if len(temp_check) > 0:
    print(f"Found {len(temp_check)} inconsistent temperature records")
else:
    print("No inconsistent temperature records found.\nThanks to @ezechiekkitwana for pointing this out.")

In [ ]:
# Verify no missing values remain
missing_after = df.isnull().sum()
if missing_after.sum() > 0:
    print("Remaining missing values:")
    print(missing_after[missing_after > 0])
else:
    print("All missing values have been successfully handled!\nThanks to @ezechiekkitwana for pointing this out.")

# Check for any infinite values created during calculations
inf_check = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
if inf_check > 0:
    print(f"Found {inf_check} infinite values that need to be addressed")
else:
    print("No infinite values found.")

In [ ]:
# Set week_start_date to datetime
df["week_start_date"] = pd.to_datetime(df["week_start_date"]).dt.date
df["week_start_date"].head()

### Feature Engineering

In [ ]:
# Create lagged features by city
for city in ['sj', 'iq']:
    city_data = df[df['city'] == city].copy()
    city_data.sort_values('week_start_date', inplace=True)
    
    # Lag the target variable (useful for autoregressive patterns)
    for lag in range(1, 9):  # 1 to 8 week lags
        city_data[f'total_cases_lag_{lag}'] = city_data['total_cases'].shift(lag)
    
    # Lag important climate variables 
    important_vars = [
        'reanalysis_specific_humidity_g_per_kg',
        'reanalysis_dew_point_temp_k',
        'station_avg_temp_c',
        'station_precip_mm',
        'reanalysis_relative_humidity_percent'
    ]
    
    for var in important_vars:
        for lag in range(1, 9):  # 1 to 8 week lags
            city_data[f'{var}_lag_{lag}'] = city_data[var].shift(lag)
    
    # Update the main dataframe
    df.loc[df['city'] == city] = city_data

In [ ]:
# Create rolling window features by city
for city in ['sj', 'iq']:
    city_data = df[df['city'] == city].copy()
    city_data.sort_values('week_start_date', inplace=True)
    
    # Variables to create rolling features for
    roll_vars = [
        'station_avg_temp_c',
        'station_precip_mm',
        'reanalysis_specific_humidity_g_per_kg',
        'reanalysis_dew_point_temp_k',
        'reanalysis_relative_humidity_percent'
    ]
    
    # Rolling means for different windows
    for var in roll_vars:
        for window in [2, 4, 8, 12]:
            city_data[f'{var}_rolling_mean_{window}w'] = city_data[var].rolling(window=window, min_periods=1).mean()
    
    # Rolling standard deviations to capture volatility
    volatility_vars = ['station_avg_temp_c', 'station_precip_mm']
    for var in volatility_vars:
        for window in [4, 8]:
            city_data[f'{var}_rolling_std_{window}w'] = city_data[var].rolling(window=window, min_periods=1).std()
    
    # Cumulative precipitation (important for breeding sites)
    for window in [4, 8, 12]:
        city_data[f'precip_cum_{window}w'] = city_data['station_precip_mm'].rolling(window=window, min_periods=1).sum()
    
    # Update the main dataframe
    df.loc[df['city'] == city] = city_data

In [ ]:
# Temperature and humidity interactions
df['temp_humidity'] = df['station_avg_temp_c'] * df['reanalysis_specific_humidity_g_per_kg']
df['temp_humidity_sq'] = df['temp_humidity'] ** 2  # Non-linear effect

# Dewpoint depression (difference between air temp and dew point)
df['dewpoint_depression'] = df['reanalysis_air_temp_k'] - df['reanalysis_dew_point_temp_k']

# Temperature range interactions
df['diurnal_temp_range_ratio'] = df['station_diur_temp_rng_c'] / df['station_avg_temp_c'].replace(0, np.nan)
df['diurnal_temp_range_ratio'] = df['diurnal_temp_range_ratio'].fillna(0)  # Replace NaN from division by zero

# Heat index (simplified version)
df['heat_index'] = df['station_avg_temp_c'] + 0.05 * df['reanalysis_relative_humidity_percent']

In [ ]:
# Average NDVI across all directions
df['ndvi_avg'] = df[['ndvi_ne', 'ndvi_nw', 'ndvi_se', 'ndvi_sw']].mean(axis=1)

# NDVI rate of change (by city)
for city in ['sj', 'iq']:
    city_data = df[df['city'] == city].copy()
    city_data.sort_values('week_start_date', inplace=True)
    
    # Calculate week-to-week changes
    city_data['ndvi_avg_change'] = city_data['ndvi_avg'].diff()
    
    # Rolling mean of NDVI
    city_data['ndvi_avg_rolling_4w'] = city_data['ndvi_avg'].rolling(window=4, min_periods=1).mean()
    
    # Update the main dataframe
    df.loc[df['city'] == city] = city_data

In [ ]:
# Calculate city-specific thresholds
for city in ['sj', 'iq']:
    city_mask = df['city'] == city
    
    # Heavy rainfall (above 90th percentile)
    rain_threshold = df.loc[city_mask, 'station_precip_mm'].quantile(0.9)
    df.loc[city_mask, 'heavy_rain'] = (df.loc[city_mask, 'station_precip_mm'] > rain_threshold).astype(int)
    
    # Extended dry period (below 10th percentile for multiple weeks)
    dry_threshold = df.loc[city_mask, 'station_precip_mm'].quantile(0.1)
    
    # Calculate dry spell duration
    city_data = df[city_mask].copy().sort_values('week_start_date')
    city_data['is_dry_week'] = (city_data['station_precip_mm'] <= dry_threshold).astype(int)
    
    # Count consecutive dry weeks
    city_data['dry_spell_count'] = 0
    current_count = 0
    
    for i, row in city_data.iterrows():
        if row['is_dry_week'] == 1:
            current_count += 1
        else:
            current_count = 0
        city_data.loc[i, 'dry_spell_count'] = current_count
    
    df.loc[city_mask, 'dry_spell_count'] = city_data['dry_spell_count']
    
    # High temperature indicator (above 90th percentile)
    temp_threshold = df.loc[city_mask, 'station_max_temp_c'].quantile(0.9)
    df.loc[city_mask, 'high_temp'] = (df.loc[city_mask, 'station_max_temp_c'] > temp_threshold).astype(int)

In [ ]:
# Create city-specific versions of important predictors
key_predictors = [
    'reanalysis_specific_humidity_g_per_kg', 
    'station_avg_temp_c',
    'reanalysis_dew_point_temp_k'
]

for var in key_predictors:
    # San Juan specific 
    df[f'sj_{var}'] = df[var] * (df['city'] == 'sj')
    
    # Iquitos specific
    df[f'iq_{var}'] = df[var] * (df['city'] == 'iq')

# One-hot encode city
df = pd.get_dummies(df, columns=['city'], prefix='city')

In [ ]:
# Temperature-related ratios
df['max_avg_temp_ratio'] = df['station_max_temp_c'] / df['station_avg_temp_c']
df['min_avg_temp_ratio'] = df['station_min_temp_c'] / df['station_avg_temp_c']

# Precipitation from different sources difference
df['precip_diff'] = df['station_precip_mm'] - df['precipitation_amt_mm']
df['precip_reanalysis_diff'] = df['station_precip_mm'] - df['reanalysis_precip_amt_kg_per_m2']

# Replace infinite values from division
for col in df.columns:
    if df[col].dtype in [np.float64, np.float32]:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        # Fill NaN with median
        df[col] = df[col].fillna(df[col].median())

In [ ]:
# For each city, calculate week-to-week differences
for city in ['sj', 'iq']:
    city_mask = df['city_' + city] == 1
    city_data = df[city_mask].copy().sort_values('week_start_date')
    
    for var in ['station_avg_temp_c', 'reanalysis_specific_humidity_g_per_kg', 'station_precip_mm']:
        # Week-to-week change
        change_col = f'{var}_change'
        city_data[change_col] = city_data[var].diff()
        
        # Rate of change (percentage)
        pct_change_col = f'{var}_pct_change'
        city_data[pct_change_col] = city_data[var].pct_change() * 100
        # Replace infinite values and NaNs
        city_data[pct_change_col] = city_data[pct_change_col].replace([np.inf, -np.inf], np.nan)
        city_data[pct_change_col] = city_data[pct_change_col].fillna(0)
        
    df.loc[city_mask] = city_data

In [ ]:
print(f"Final shape after clean Feature Engineering: {df.shape}")
df.tail(3)